# Selenium 2026 — dynamic websites
**Optional Week 5 extension · DE0823 · RTU**

30–45 minutes. Requests retrieves responses; Beautiful Soup parses markup; Selenium controls a real browser. Use the simplest adequate method. Our [dynamic archive](https://valrcs.github.io/Digital_Discourse_ETH713_Fall_2020/playground/dynamic-archive/index.html) is synthetic and explicitly intended for bounded practice. No SS.com requests. Use a local browser first; Colab requires a compatible browser and driver, not only pip packages. Set ARCHIVE_URL to a locally served checkout to avoid depending on Pages.


## 1. Setup
Install packages into the current kernel. Selenium Manager normally locates or downloads a matching driver if permitted. A browser must be available. Headed mode shows the browser; headless runs without a window. Set CHROME_BINARY or CHROMEDRIVER environment variables if a managed environment provides explicit paths. Colab is not guaranteed: check browser/driver availability. The no-sandbox option is off by default; only opt in if needed in an isolated cloud runtime, not on a regular workstation.


In [ ]:
%pip install selenium beautifulsoup4 requests


In [ ]:
import os, csv, json, shutil
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
import requests, selenium
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

PAGE_URL = os.environ.get("ARCHIVE_URL", "https://valrcs.github.io/Digital_Discourse_ETH713_Fall_2020/playground/dynamic-archive/index.html")
JSON_URL = urljoin(PAGE_URL, "archive.json")
HEADLESS = True  # False for a visible local browser
ISOLATED_CLOUD_NO_SANDBOX = False  # Only opt in for an isolated runtime if required
SELECTOR = "article.archive-record"
TIMEOUT = 15
print("Selenium:", selenium.__version__)
print("Browser candidates:", [p for p in (shutil.which("google-chrome"), shutil.which("chromium")) if p])

def make_driver():
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1280,900")
    if ISOLATED_CLOUD_NO_SANDBOX:
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
    if os.environ.get("CHROME_BINARY"):
        options.binary_location = os.environ["CHROME_BINARY"]
    if os.environ.get("CHROMEDRIVER"):
        return webdriver.Chrome(service=Service(os.environ["CHROMEDRIVER"]), options=options)
    return webdriver.Chrome(options=options)


## 2. Compare initial HTML with direct JSON
The HTML response contains the shell, not the records. Check HTTP errors. The public JSON endpoint is the simpler way to retrieve all nine records.


In [ ]:
response = requests.get(PAGE_URL, timeout=20)
response.raise_for_status()
static_html = response.text
static_soup = BeautifulSoup(static_html, "html.parser")
assert len(static_soup.select(SELECTOR)) == 0
print("Static records:", len(static_soup.select(SELECTOR)))
response = requests.get(JSON_URL, timeout=20)
response.raise_for_status()
api_records = response.json()
assert len(api_records) == 9 and len({r["id"] for r in api_records}) == 9
print("Direct JSON records:", len(api_records))


## 3. First browser session
find_element returns a WebElement or raises; find_elements returns a list (possibly empty). .text is rendered text; get_dom_attribute reads markup attributes. Each browser session is self-contained and closes in finally.


In [ ]:
driver = make_driver()
try:
    driver.get(PAGE_URL)
    print(driver.title, driver.current_url)
    print("Browser:", driver.capabilities.get("browserVersion"))
    assert len(driver.find_elements(By.CSS_SELECTOR, SELECTOR)) == 0
finally:
    driver.quit()


## 4. Explicit waits, interaction and hybrid extraction
Wait for the first batch of three, then six and nine. Explicit waits poll; the deliberate delay is 900 ms, but do not use guessed sleeps. Compare direct WebElements, Beautiful Soup parsed snapshot and JSON. Re-locate elements after DOM updates. Filters only affect loaded records. Rīg matches two titles by literal substring, not lemmatization. If an assertion fails the browser closes; no synthetic fallback masks errors.


In [ ]:
def wait_count(driver, expected):
    return WebDriverWait(driver, TIMEOUT).until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, SELECTOR)) == expected
        and d.find_element(By.ID, "archive-status").get_dom_attribute("data-state") in ("ready", "idle")
    )

def click(driver, element_id):
    WebDriverWait(driver, TIMEOUT).until(
        EC.element_to_be_clickable((By.ID, element_id))
    ).click()

def from_elements(driver):
    return [{
        "id": e.get_dom_attribute("data-id"),
        "title": e.find_element(By.CSS_SELECTOR, "h2").text,
        "year": int(e.get_dom_attribute("data-year")),
        "category": e.get_dom_attribute("data-category"),
    } for e in driver.find_elements(By.CSS_SELECTOR, SELECTOR)]

def from_html(html):
    return [{
        "id": a["data-id"],
        "title": a.select_one("h2").get_text(strip=True),
        "year": int(a["data-year"]),
        "category": a["data-category"],
    } for a in BeautifulSoup(html, "html.parser").select(SELECTOR)]

# Avoid stale output after an unsuccessful rerun.
records = None
actions = []
driver = make_driver()
try:
    driver.get(PAGE_URL)
    actions.append("open archive")
    wait_count(driver, 0)
    click(driver, "load-collection")
    wait_count(driver, 3)
    actions.append("load collection; wait for 3")
    for expected in (6, 9):
        click(driver, "load-more")
        wait_count(driver, expected)
        actions.append(f"load more; wait for {expected}")
    assert not driver.find_element(By.ID, "load-more").is_enabled()
    direct = from_elements(driver)
    rendered_html = driver.page_source
    candidate = from_html(rendered_html)
    assert candidate == direct == api_records
    assert len({r["id"] for r in candidate}) == 9
    browser_version = driver.capabilities.get("browserVersion")
    retrieved_at = datetime.now(timezone.utc).isoformat()
    print("Selenium and hybrid records:", len(candidate))
    Select(driver.find_element(By.ID, "archive-category")).select_by_visible_text("Research")
    wait_count(driver, 3)
    Select(driver.find_element(By.ID, "archive-category")).select_by_visible_text("All")
    wait_count(driver, 9)
    driver.find_element(By.ID, "archive-search").send_keys("Rīg")
    wait_count(driver, 2)
    click(driver, "archive-reset")
    wait_count(driver, 0)
    records = candidate
    print("Filter, search and reset verified.")
finally:
    driver.quit()


## 5. Export with provenance
Capture the snapshot before filtering. page_source is current document serialization, not original response bytes. Preserve both snapshots. The records are invented. Export requires a successful browser workflow.


In [ ]:
assert records is not None and len(records) == 9 and records == api_records
out = Path("outputs/selenium_2026")
out.mkdir(parents=True, exist_ok=True)
metadata = {
    "source_kind": "synthetic_teaching_archive",
    "source_url": PAGE_URL,
    "json_url": JSON_URL,
    "retrieved_at_utc": retrieved_at,
    "method": "selenium_then_beautifulsoup",
    "selenium_version": selenium.__version__,
    "browser_version": browser_version,
    "actions": actions,
    "selector": SELECTOR,
    "filters": {"category": "All", "search": ""},
    "records_extracted": len(records),
}
(out / "records.json").write_text(json.dumps({"metadata": metadata, "records": records}, ensure_ascii=False, indent=2), encoding="utf-8")
(out / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
with (out / "records.csv").open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["id", "title", "year", "category"])
    writer.writeheader()
    writer.writerows(records)
(out / "initial.html").write_text(static_html, encoding="utf-8")
(out / "rendered.html").write_text(rendered_html, encoding="utf-8")
print(out.resolve())


## 6. Exercises and limitations
1. Inspect View Source and Elements after loading.
2. Inspect archive.json in Network.
3. Filter Language (two after full load).
4. Explain why direct JSON is preferable here.
5. Try an invalid locator and diagnose a timeout within try/finally.

Respect site policies, authorization, privacy and rate limits. Do not circumvent authentication or CAPTCHAs. Shadow DOM, iframes and canvas require special handling. See [Selenium docs](https://www.selenium.dev/documentation/), [waits](https://www.selenium.dev/documentation/webdriver/waits/), [locators](https://www.selenium.dev/documentation/webdriver/elements/finders/) and [Selenium Manager](https://www.selenium.dev/documentation/selenium_manager/).
